Updated inference defaults: local labels use empirical adjusted p-values; zero permutations are untested. ICG uses within-type contact flags and an explicit expression scale. Recorded outputs were cleared and require rerunning. See `docs/analysis.md` for assumptions.


# `tl.ligrec` vs `squidpy.gr.ligrec`

Astrolabe's CPU `ligrec` restricts every score to actual spatial-graph edges
(sender A, receiver B connected in the graph); squidpy's `ligrec` is a
CellPhoneDB-style test over **all** cells of each type, with no spatial
restriction. To compare the same statistic rather than two different ones,
astrolabe is run here on a **complete graph**: every cell of type A is
paired with every cell of type B, so astrolabe's per-edge mean reduces to
the same cluster-level quantities squidpy computes directly.

`tl.ligrec` now uses the CellPhoneDB averaging convention,
`(mean(L) + mean(R)) / 2`, matching squidpy's own formula (it previously
scored each edge as a product, `L_i * R_j`). On the complete graph this
makes `sender != receiver` an exact algebraic match; `A == B` is still
reported separately, since astrolabe's edge set excludes self-pairs
(`i != j`), which squidpy's cluster-mean formula does not.


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import time
import numpy as np
import astrolabe as ab
import squidpy as sq

mdata = ab.datasets.xenium_tiny()
ab.pipeline.filter_xenium(mdata)
ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)
ab.pp.highly_variable_genes(mdata)
ab.tl.pca(mdata)
ab.tl.transcriptional_neighbors(mdata)
ab.tl.cluster(mdata, resolution=0.5)
ab.tl.spatial_neighbors(mdata)

adata = mdata["xenium"]

# Fairness step: copy astrolabe's own spatial graph into the obsp key
# squidpy expects, so both tools permute/aggregate over the *identical*
# graph topology rather than two independently-built ones.
adata.obsp["spatial_connectivities"] = adata.obsp["spatial_neighbors_connectivities"].copy()
adata


## Dataset and a shared graph

`ab.datasets.xenium_tiny()`, QC'd, normalized, clustered (Leiden), with
astrolabe's own spatial graph copied into `obsp['spatial_connectivities']` -
the key `squidpy.gr.*` looks for by default. Without this step the two tools
would each build their own KNN graph independently and any disagreement
could just be graph-construction noise, not a real difference in the
statistic itself.


## A complete graph, for a fair comparison

Astrolabe's real spatial graph only connects nearby cells; squidpy's formula
has no such restriction. Writing an all-pairs (minus self-loops)
connectivity matrix into a separate `neighbors_key='complete'` slot lets
astrolabe answer the same spatially-unrestricted question squidpy always
asks, rather than comparing a local statistic to a global one.


In [ ]:
import scipy.sparse as sp

n = adata.n_obs
complete_adj = sp.csr_matrix(np.ones((n, n)) - np.eye(n))
adata.obsp["complete_connectivities"] = complete_adj
adata.uns["complete"] = {"connectivities_key": "complete_connectivities", "distances_key": "complete_distances"}


## Astrolabe: `tl.ligrec` on the complete graph

A small curated ligand-receptor gene list, intersected against the panel's
actual `var_names` at runtime (Xenium panels vary by dataset).

One gate difference matters here: astrolabe applies the CellPhoneDB
expressing-fraction gate to **both** `means` and `pvalues` (untested
entries are NaN), while squidpy's `threshold=` masks only `pvalues` and
always reports every mean. On this tiny sparse panel almost no cluster
expresses these genes in more than 10% of cells, so astrolabe's default
gate would NaN every mean and leave nothing to compare. The gate is
therefore disabled (`expr_frac_threshold=0.0`) — this benchmark compares
the *scoring convention*, not the gating policy.

In [ ]:
import pandas as pd

candidate_pairs = pd.DataFrame({
    "ligand":   ["CCL19", "EDN1",  "CD86", "CD86"],
    "receptor": ["CCR7",  "EDNRB", "CD28", "CTLA4"],
})
lr_pairs = candidate_pairs[
    candidate_pairs["ligand"].isin(adata.var_names) & candidate_pairs["receptor"].isin(adata.var_names)
].reset_index(drop=True)
print(f"{len(lr_pairs)} / {len(candidate_pairs)} candidate LR pairs present in this panel")
lr_pairs


In [ ]:
_t0 = time.perf_counter()
# expr_frac_threshold=0.0: disable the expressing-fraction gate (see above) so
# every mean is computed, matching squidpy, which never gates its means.
ab.tl.ligrec(mdata, "leiden", lr_pairs, neighbors_key="complete", n_perms=200, seed=0,
             expr_frac_threshold=0.0)
t_astro_s = time.perf_counter() - _t0
print(f"astrolabe runtime: {t_astro_s:.4f}s")
means_astro = adata.uns["ligrec"]["means"]
pvalues_astro = adata.uns["ligrec"]["pvalues"]
means_astro.head()


## Reference: `squidpy.gr.ligrec`

squidpy expects `interactions=` with `'source'`/`'target'` columns (the same
rename astrolabe's own GPU path already performs when delegating to
`rsc.gr.ligrec`).


In [ ]:
interactions = lr_pairs.rename(columns={"ligand": "source", "receptor": "target"})
_t0 = time.perf_counter()
result_sq = sq.gr.ligrec(
    adata, cluster_key="leiden", interactions=interactions,
    n_perms=200, seed=0, threshold=0.05, copy=True, show_progress_bar=False,
    use_raw=False,
)
t_ref_s = time.perf_counter() - _t0
print(f"reference runtime: {t_ref_s:.4f}s")
means_ref = result_sq["means"]
pvalues_ref = result_sq["pvalues"]
means_ref.head()


## Agreement

Restricted to `sender != receiver` pairs, per the algebraic argument above -
`sender == receiver` is expected to diverge for the documented self-pair
reason, and is reported separately rather than folded into the headline
number.


In [ ]:
rows = []
for (ligand, receptor) in means_astro.index:
    for (sender, receiver) in means_astro.columns:
        if (ligand, receptor) not in means_ref.index or (sender, receiver) not in means_ref.columns:
            continue
        rows.append({
            "ligand": ligand, "receptor": receptor, "sender": sender, "receiver": receiver,
            "mean_astro": means_astro.loc[(ligand, receptor), (sender, receiver)],
            "mean_ref": means_ref.loc[(ligand, receptor), (sender, receiver)],
            "same_type": sender == receiver,
        })
merged = pd.DataFrame(rows).dropna(subset=["mean_astro", "mean_ref"])

cross = merged[~merged["same_type"]]
same = merged[merged["same_type"]]

assert len(cross) > 0, "empty comparison set — gating mismatch between the two tools?"
r_cross = np.corrcoef(cross["mean_astro"], cross["mean_ref"])[0, 1]
max_diff_cross = float((cross["mean_astro"] - cross["mean_ref"]).abs().max())
print(f"sender != receiver: Pearson r = {r_cross:.6f}, max |diff| = {max_diff_cross:.2e}  (n={len(cross)})")

if len(same) > 0:
    r_same = np.corrcoef(same["mean_astro"], same["mean_ref"])[0, 1]
    print(f"sender == receiver: Pearson r = {r_same:.4f}  (n={len(same)}, reported separately, see note above)")

print(f"Runtime: astrolabe {t_astro_s:.4f}s vs squidpy {t_ref_s:.4f}s "
      f"({max(t_astro_s, t_ref_s) / min(t_astro_s, t_ref_s):.1f}x astrolabe {'faster' if t_astro_s < t_ref_s else 'slower'}) -- "
      f"astrolabe's call here runs on the injected complete graph (n*(n-1) edges), a much bigger "
      f"edge set than its real spatial graph would have, so this isn't astrolabe's typical speed.")

panel = ab.pl.scatter(
    adata, x=cross["mean_astro"].to_numpy(), y=cross["mean_ref"].to_numpy(), annotate_corr=True,
    xlabel="astrolabe tl.ligrec on complete graph (means)", ylabel="squidpy.gr.ligrec (means)",
    title="Ligand-receptor mean score agreement, sender != receiver",
)
panel.show()


## Interpretation

`sender != receiver` should be a strong, near-exact match: astrolabe's
`(mean(L) + mean(R)) / 2` score, computed over every edge of the complete
graph, reduces algebraically to the same per-cluster means squidpy's own
CellPhoneDB formula uses. A small residual gap can remain from details like
raw-vs-normalized input or small-cluster handling. `sender == receiver` is
reported separately and is expected to diverge more, since astrolabe's
edge-based formula never pairs a cell with itself, while squidpy's
cluster-mean formula has no such exclusion.
